# Salience-Sensitive Image Generation

Generates 50 images per prompt for each of:
1. DDPM baseline
2. ScoreNormPhi
3. DiversityPhi
4. ScoreAlignmentPhi

Prompts: llama, wolf, monkey, butterfly

Outputs saved to `../outputs/sd_experiment/<method>/<prompt>/`

In [ ]:
import torch
from pathlib import Path
from diffusers import StableDiffusionPipeline

from sd.pipeline import SalienceGradSDPipeline
from sd.phi_sd import ScoreNormPhiSD, DiversityPhiSD, ScoreAlignmentPhiSD

DEVICE     = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
MODEL_ID   = "runwayml/stable-diffusion-v1-5"
OUTPUT_DIR = Path("../outputs/sd_experiment")
PROMPTS    = ["llama", "wolf", "monkey", "butterfly"]

NUM_IMAGES = 50
NUM_STEPS  = 500
CFG_SCALE  = 5.0
SAL_SCALE  = 1.0
BATCH_SIZE = 8
SEED       = 2024

print(f"Device: {DEVICE}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# ---------------------------------------------------------------------------
# Load salience pipeline and generate null embeddings
# ---------------------------------------------------------------------------
pipe = SalienceGradSDPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
).to(DEVICE)
pipe.set_progress_bar_config(disable=False)

null_tokens = pipe.tokenizer(
    [""],
    padding="max_length",
    max_length=pipe.tokenizer.model_max_length,
    return_tensors="pt",
).input_ids.to(DEVICE)

with torch.no_grad():
    null_embeds = pipe.text_encoder(null_tokens).last_hidden_state

print(f"Pipeline loaded. Null embeds shape: {null_embeds.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Helper: generate N images for a prompt in batches
# ---------------------------------------------------------------------------
def generate_images(pipe, prompt, save_dir, n=50, batch_size=8, seed=2024):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    generated = 0
    batch_idx = 0
    while generated < n:
        current_batch = min(batch_size, n - generated)
        generator = torch.Generator(device=DEVICE).manual_seed(seed + batch_idx)
        pipe(
            prompt=prompt,
            num_images_per_prompt=current_batch,
            num_inference_steps=NUM_STEPS,
            guidance_scale=CFG_SCALE,
            generator=generator,
            save_dir=save_dir,
            offset=generated,
        )
        generated += current_batch
        batch_idx += 1
        print(f"    {generated}/{n}")

## 1. DDPM Baseline

In [ ]:
print("=== DDPM Baseline ===")

baseline_pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
).to(DEVICE)
baseline_pipe.set_progress_bar_config(disable=False)

for prompt in PROMPTS:
    print(f"\nPrompt: {prompt}")
    save_dir = OUTPUT_DIR / "baseline" / prompt
    save_dir.mkdir(parents=True, exist_ok=True)
    generated = 0
    batch_idx = 0
    while generated < NUM_IMAGES:
        current_batch = min(BATCH_SIZE, NUM_IMAGES - generated)
        generator = torch.Generator(device=DEVICE).manual_seed(SEED + batch_idx)
        images = baseline_pipe(
            prompt=prompt,
            num_images_per_prompt=current_batch,
            num_inference_steps=NUM_STEPS,
            guidance_scale=CFG_SCALE,
            generator=generator,
        ).images
        for idx, img in enumerate(images):
            img.save(save_dir / f"{generated + idx}.png")
        generated += current_batch
        batch_idx += 1
        print(f"  {generated}/{NUM_IMAGES}")

del baseline_pipe
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print("\nBaseline done.")

## 2. ScoreNormPhi

In [ ]:
print("=== ScoreNormPhi ===")

phi_norm = ScoreNormPhiSD(conditional=False)
pipe.setup_phi(phi_norm, context={"null_embeds": null_embeds})
pipe.set_salience_scale(SAL_SCALE)
pipe.set_guidance_frequency(1)

for prompt in PROMPTS:
    print(f"\nPrompt: {prompt}")
    generate_images(
        pipe, prompt,
        save_dir=OUTPUT_DIR / "scorenorm" / prompt,
        n=NUM_IMAGES, batch_size=BATCH_SIZE, seed=SEED,
    )

print("\nScoreNormPhi done.")

## 3. DiversityPhi

In [ ]:
print("=== DiversityPhi ===")

phi_div = DiversityPhiSD()
pipe.setup_phi(phi_div, context={})
pipe.set_salience_scale(SAL_SCALE)
pipe.set_guidance_frequency(1)

for prompt in PROMPTS:
    print(f"\nPrompt: {prompt}")
    generate_images(
        pipe, prompt,
        save_dir=OUTPUT_DIR / "diversity" / prompt,
        n=NUM_IMAGES, batch_size=BATCH_SIZE, seed=SEED,
    )

print("\nDiversityPhi done.")

## 4. ScoreAlignmentPhi

In [ ]:
print("=== ScoreAlignmentPhi ===")

phi_align = ScoreAlignmentPhiSD(conditional=False)
pipe.setup_phi(phi_align, context={"null_embeds": null_embeds})
pipe.set_salience_scale(SAL_SCALE)
pipe.set_guidance_frequency(1)

for prompt in PROMPTS:
    print(f"\nPrompt: {prompt}")
    generate_images(
        pipe, prompt,
        save_dir=OUTPUT_DIR / "scorealignment" / prompt,
        n=NUM_IMAGES, batch_size=BATCH_SIZE, seed=SEED,
    )

print("\nScoreAlignmentPhi done.")

## Done

All images saved to `../outputs/sd_experiment/`. Run the evaluation notebook next.